# Task 1 - Step 7: Export results

Bookkeeping only (no new computation):

1. Copy all Task 1 tables to top-level `results/task1/` and all figures to `figures/task1/`
   (figure names get a `task1_` prefix if missing).
2. Build one consolidated nested `results/task1/results.json` with raw, unrounded values and full
   per-epoch head-training curves. Any metric that could not be computed is `null` with a note.

In [1]:
# ---- Common header (identical in every Task 1 notebook) ----
import json, random, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# Locate the task1/ folder no matter where Jupyter was launched from.
ROOT = Path.cwd()
if not (ROOT / "configs").exists() and (ROOT / "task1" / "configs").exists():
    ROOT = ROOT / "task1"

CFG = json.loads((ROOT / "configs" / "task1_config.json").read_text())
SEED = CFG["seed"]

# Small, git-tracked outputs -> results/.  Large tensors/weights/raw data -> cache/ and data/ (git-ignored).
RES = ROOT / "results"
CACHE = ROOT / "cache"
DATA = ROOT / "data"
for p in [RES / "subset", RES / "cue_conflicts", RES / "tables", RES / "figures",
          CACHE / "sets", CACHE / "features", CACHE / "heads", CACHE / "weights", DATA]:
    p.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


def set_seed(seed=SEED):
    """Fix every RNG we use so reruns reproduce the same numbers."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed()
print("ROOT:", ROOT, "| device:", DEVICE, "| seed:", SEED)

ROOT: C:\Users\afifh\Desktop\ATML\PA1\task1 | device: cuda | seed: 6304


In [2]:
import shutil, math

REPO = ROOT.parent
R_OUT, F_OUT = REPO / "results" / "task1", REPO / "figures" / "task1"
R_OUT.mkdir(parents=True, exist_ok=True); F_OUT.mkdir(parents=True, exist_ok=True)
TAB, FIG = RES / "tables", RES / "figures"

B = torch.load(CACHE / "outputs" / "task1_outputs.pt")
assert not B["smoke"], "outputs are from SMOKE mode - rerun notebook 04 without TASK1_SMOKE before exporting"

copied = []
for p in sorted(TAB.glob("*")) + [RES / "design_choices.json", RES / "subset" / "subset_ids.json",
                                  RES / "subset" / "head_split.json", RES / "subset" / "patch_permutations.json",
                                  RES / "cue_conflicts" / "metadata.csv"]:
    if p.is_file() and not p.name.startswith("."):
        dst = R_OUT / (p.name if p.name.startswith("task1_") else f"task1_{p.name}")
        shutil.copy2(p, dst); copied.append(dst.name)
for p in sorted(FIG.glob("*")):
    if p.suffix in {".png", ".pdf"}:
        dst = F_OUT / (p.name if p.name.startswith("task1_") else f"task1_{p.name}")
        shutil.copy2(p, dst); copied.append("figures/" + dst.name)
print(len(copied), "files copied")

49 files copied


In [3]:
def clean_nan(o):
    if isinstance(o, float) and (math.isnan(o) or math.isinf(o)):
        return None
    if isinstance(o, dict):
        return {k: clean_nan(v) for k, v in o.items()}
    if isinstance(o, list):
        return [clean_nan(v) for v in o]
    return o


def rj(name):
    return json.loads((TAB / name).read_text())


head = rj("task1_head_training_log.json")
res = {
    "task": "task1_inductive_biases",
    "seed": SEED, "config": CFG,
    "design_choices": json.loads((RES / "design_choices.json").read_text()),
    "subset": {"n_images": int(len(B["labels"])), "per_class": dict(zip(B["classes"], np.bincount(B["labels"].numpy()).tolist())),
               "ids_file": "results/task1/task1_subset_ids.json"},
    "head_training": head,
    "clip_prompts": rj("task1_clip_prompts.json"),
    "clean": rj("task1_clean_metrics.json")["metrics"],
    "grayscale": rj("task1_grayscale_metrics.json")["metrics"],
    "hue_rotate": rj("task1_hue_metrics.json")["metrics"],
    "patch_shuffle": rj("task1_patchshuffle_metrics.json")["metrics"],
}

acc = pd.read_csv(TAB / "cue_conflict_acceptance.csv")
sb = pd.read_csv(TAB / "task1_shape_texture_results.csv")
res["cue_conflict"] = {"acceptance": {"per_cell": acc.to_dict("records"),
                                      "total_generated": int(acc.generated.sum()), "total_accepted": int(acc.accepted.sum()),
                                      "total_rejected": int(acc.rejected.sum()), "total_kept": int(acc.kept.sum())}}
for s in B["systems"]:
    g = sb[sb.system == s].drop(columns="system").set_index("scope")
    res["cue_conflict"][s] = {"all": g.loc["all"].to_dict(),
                              "by_scope": {k: v for k, v in g.to_dict("index").items() if k != "all"}}

trd = pd.read_csv(TAB / "task1_translation_results.csv")
res["translation"] = {}
for s in B["systems"]:
    res["translation"][s] = {}
    for d, g in trd[trd.system == s].groupby("displacement_px"):
        m = g[g.direction == "mean"].iloc[0]
        res["translation"][s][f"delta_{d}"] = {
            "accuracy": m.top1_acc, "accuracy_std_over_directions": m.top1_acc_std,
            "consistency": m.consistency, "consistency_std_over_directions": m.consistency_std,
            "per_direction": {r.direction: {"accuracy": r.top1_acc, "consistency": r.consistency}
                              for r in g[g.direction != "mean"].itertuples()}}

sim = pd.read_csv(TAB / "task1_representation_similarity.csv")
res["representation_similarity"] = {bb: g.drop(columns="backbone").set_index("intervention").to_dict("index")
                                    for bb, g in sim.groupby("backbone")}
pvr = pd.read_csv(TAB / "task1_prediction_vs_representation.csv")
res["prediction_vs_representation"] = {s: g.drop(columns=["system"]).set_index("intervention").to_dict("index")
                                       for s, g in pvr.groupby("system")}
res["clip_zeroshot_vs_head"] = pd.read_csv(TAB / "task1_clip_zeroshot_vs_head.csv").set_index("set").to_dict("index")
res["clean_neighbour_agreement"] = {bb: g.drop(columns="backbone").set_index("condition").to_dict("index")
                                    for bb, g in pd.read_csv(TAB / "task1_clean_neighbour_agreement.csv").groupby("backbone")}
res["_notes"] = {"null_values": "null = not computable (e.g. shape bias with zero shape+texture decisions, "
                                "AUROC when all predictions are stable, or no errors to average confidence over)."}
res["files"] = {"tables": sorted(p.name for p in R_OUT.glob("*") if p.is_file()),
                "figures": sorted(p.name for p in F_OUT.glob("*"))}

(R_OUT / "results.json").write_text(json.dumps(clean_nan(res), indent=1))
print("wrote", R_OUT / "results.json")

wrote C:\Users\afifh\Desktop\ATML\PA1\results\task1\results.json


In [4]:
# Short summary for visual confirmation
print("clean top-1:", {s: round(res["clean"][s]["top1_acc"], 4) for s in B["systems"]})
print("grayscale d-acc:", {s: round(res["grayscale"][s]["delta_acc_vs_clean"], 4) for s in B["systems"]})
print("hue d-acc:", {s: round(res["hue_rotate"][s]["delta_acc_vs_clean"], 4) for s in B["systems"]})
print("patch d-acc:", {s: round(res["patch_shuffle"][s]["delta_acc_vs_clean"], 4) for s in B["systems"]})
print("shape bias / coverage:", {s: (round(res["cue_conflict"][s]["all"]["shape_bias_pct"], 1),
                                      round(res["cue_conflict"][s]["all"]["coverage_pct"], 1)) for s in B["systems"]})
print("translation acc @32:", {s: round(res["translation"][s]["delta_32"]["accuracy"], 4) for s in B["systems"]})

clean top-1: {'resnet50_head': 0.97, 'vit_b16_head': 0.968, 'clip_head': 0.968, 'clip_zeroshot': 0.936}
grayscale d-acc: {'resnet50_head': -0.022, 'vit_b16_head': -0.022, 'clip_head': -0.04, 'clip_zeroshot': -0.028}
hue d-acc: {'resnet50_head': -0.03, 'vit_b16_head': -0.036, 'clip_head': -0.05, 'clip_zeroshot': -0.02}
patch d-acc: {'resnet50_head': -0.084, 'vit_b16_head': -0.054, 'clip_head': -0.172, 'clip_zeroshot': -0.138}
shape bias / coverage: {'resnet50_head': (53.0, 52.8), 'vit_b16_head': (74.3, 56.0), 'clip_head': (88.7, 56.4), 'clip_zeroshot': (87.6, 54.8)}
translation acc @32: {'resnet50_head': np.float64(0.957), 'vit_b16_head': np.float64(0.963), 'clip_head': np.float64(0.9455), 'clip_zeroshot': np.float64(0.9275)}
